# BIOT 6900 · Module 2 — Multi-Omics Target Identification & Validation
### Week 2 · Computational Lab 2 · Starter Notebook

**What this notebook does.** You'll integrate three data layers — transcriptomics (RNA), proteomics (protein), and genomics — to nominate and *rank* disease targets, then hand that ranked list to Weeks 3–4.

**How today is built — I do → we do → you do:**
- **Part 1 (worked, already complete):** CPTAC **breast cancer** on *synthetic* data. The tumors are measured at all three layers, so the data is *sample-matched* — you integrate **at the sample level**. Your instructor runs these cells; follow along.
- **Part 2 (together, in class):** find and load *real* CPTAC data, then re-run Part 1 on real numbers. Not graded.
- **Part 3 (your assignment, `# TODO` cells):** **Alzheimer's disease**. The cohorts are *not* matched across layers, so you integrate **at the gene level**. You transfer the Part 1 method to messier, more realistic data.

> **The one idea to carry through:** *how* you integrate is dictated by *what your data has matched.* Matched → correlate within samples. Unmatched → compare gene-level summaries. Same goal, different machinery.

**Data.** Part 1 looks for the synthetic CPTAC files in `data/` and, if they're absent, falls back to clearly-labelled demo data so it always runs. Part 3 requires the three Alzheimer's matrices posted on **Canvas** — place them in `data/` (see the Lab Guide).

*Continuity from Module 1:* you'll see **TP53 / p53** (`P04637`, PDB `1TUP`) surface in the cancer half and **APOE** (`rs7412`) in the Alzheimer's half.

**Before you submit:** `Kernel → Restart & Run All` so the whole notebook executes top to bottom.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(6900)  # fixed seed so results are reproducible

# Module 1 continuity anchors
P53_UNIPROT, P53_PDB = "P04637", "1TUP"   # p53  -> cancer half
APOE_VARIANT = "rs7412"                    # APOE -> Alzheimer's half

# Default scoring weights for this course: equal across the three layers.
# You MAY change these in Part 3 — but if you do, you must justify it in your report.
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

## Helper functions you'll use in both parts
Two small functions do the scoring work. `rank_percentile` puts any layer's scores on a common 0–1 scale (by rank, so it's robust to outliers and to layers being on different units). `multi_evidence_score` normalizes each layer and takes the weighted sum.

In [2]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## Data loading (provided — you don't need to edit these)
`load_cptac()` reads the matched breast-cancer matrices for Part 1 (or synthesizes demo data if they're missing). `load_ad()` reads the three Alzheimer's matrices you download from Canvas for Part 2.

In [3]:
CPTAC_GENES = ["TP53", "PIK3CA", "ERBB2", "ESR1", "GATA3", "MYC", "CDH1",
               "MAP3K1", "PTEN", "AKT1", "RB1", "CCND1", "FOXA1", "MKI67",
               "EGFR", "BRCA1", "BRCA2", "KRT5", "VIM", "ACTB"]


def _synth_cptac(n_tumor=60, n_normal=15):
    genes = CPTAC_GENES
    tumor_ids = [f"T{i:02d}" for i in range(n_tumor)]
    normal_ids = [f"N{i:02d}" for i in range(n_normal)]
    coupling = pd.Series(RNG.uniform(0.05, 0.80, len(genes)), index=genes)

    def gene_matrix(sample_ids, shift):
        rna = pd.DataFrame(RNG.normal(0, 1, (len(genes), len(sample_ids))),
                           index=genes, columns=sample_ids).add(shift, axis=0)
        noise = pd.DataFrame(RNG.normal(0, 1, rna.shape), index=genes, columns=sample_ids)
        prot = rna.mul(coupling, axis=0) + noise.mul(1 - coupling + 0.35, axis=0)
        return rna, prot

    shift = pd.Series(0.0, index=genes)
    for g in ["TP53", "ERBB2", "MKI67", "MYC", "PIK3CA"]:
        shift[g] = RNG.uniform(1.2, 2.2)
    rna_t, prot_t = gene_matrix(tumor_ids, shift)
    rna_n, prot_n = gene_matrix(normal_ids, pd.Series(0.0, index=genes))
    rna = pd.concat([rna_t, rna_n], axis=1)
    prot = pd.concat([prot_t, prot_n], axis=1)
    mut_freq = pd.Series(RNG.uniform(0.0, 0.05, len(genes)), index=genes)
    for g in ["TP53", "PIK3CA", "CDH1", "GATA3"]:
        mut_freq[g] = RNG.uniform(0.25, 0.45)
    meta = pd.Series(["tumor"] * n_tumor + ["normal"] * n_normal,
                     index=tumor_ids + normal_ids, name="group")
    return rna, prot, mut_freq, meta


def load_cptac():
    p = {"rna": "data/cptac_brca_rna.tsv", "prot": "data/cptac_brca_protein.tsv",
         "mut": "data/cptac_brca_mutation.tsv"}
    if all(os.path.exists(v) for v in p.values()):
        rna = pd.read_csv(p["rna"], sep="\t", index_col=0)
        prot = pd.read_csv(p["prot"], sep="\t", index_col=0)
        mut = pd.read_csv(p["mut"], sep="\t", index_col=0).iloc[:, 0]
        print("Loaded CPTAC files from data/.")
        return rna, prot, mut, None
    print("!! WARNING: CPTAC files not found -> SYNTHETIC demo data "
          "(illustrative only, not real CPTAC values).")
    return _synth_cptac()


def load_ad():
    p = {"tx": "data/ad_transcriptomics.tsv", "pr": "data/ad_proteomics.tsv",
         "gw": "data/ad_gwas.tsv"}
    missing = [v for v in p.values() if not os.path.exists(v)]
    if missing:
        raise FileNotFoundError(
            "Alzheimer's matrices not found: " + ", ".join(missing) +
            "\nDownload the three files from Canvas and put them in a 'data/' folder "
            "next to this notebook (see the Lab Guide).")
    return (pd.read_csv(p["tx"], sep="\t"),
            pd.read_csv(p["pr"], sep="\t"),
            pd.read_csv(p["gw"], sep="\t"))

---
# Part 1 — Worked example: CPTAC breast cancer *(run and read; nothing to edit)*

**Why integrate at all?** Each layer can produce artifacts the others don't share, so a target supported across genome, transcriptome, and proteome is a stronger bet than one seen in only one layer. Integration is triangulation.

Because CPTAC measures the **same tumors** at every layer, we can line samples up and integrate **at the sample level.**

In [4]:
rna, prot, mut_freq, meta = load_cptac()
print("RNA matrix:    ", rna.shape, "(genes x samples)")
print("Protein matrix:", prot.shape)
print("Mutation freq: ", mut_freq.shape)

Loaded CPTAC files from data/.
RNA matrix:     (23121, 122) (genes x samples)
Protein matrix: (12621, 122)
Mutation freq:  (9448,)


### 1.2 Harmonize identifiers → a common gene key
The real technical wall in multi-omics: gene symbols, Ensembl IDs, and UniProt accessions don't line up for free. Here we intersect on a shared key and check how many genes *survive the join* — always your first reality check.

In [5]:
common = rna.index.intersection(prot.index).intersection(mut_freq.index)
print(f"Genes surviving the 3-way join: {len(common)} "
      f"(RNA {len(rna.index)}, protein {len(prot.index)}, mutation {len(mut_freq.index)})")
rna, prot, mut_freq = rna.loc[common], prot.loc[common], mut_freq.loc[common]
samples = rna.columns.intersection(prot.columns)

Genes surviving the 3-way join: 6306 (RNA 23121, protein 12621, mutation 9448)


### 1.3–1.4 Sample-level RNA–protein correlation
With matched samples we can ask, per gene, *does protein track RNA across patients?* The answer is usually **partial** — correlation is modest and varies by gene. A gene where protein does **not** track RNA isn't broken data; it's a signal of post-transcriptional regulation (translational buffering, protein turnover).

In [6]:
corr = pd.Series(
    {g: stats.spearmanr(rna.loc[g, samples], prot.loc[g, samples]).statistic
     for g in common}, name="rna_prot_corr")
print(f"RNA-protein correlation: median={corr.median():.2f}, "
      f"range=[{corr.min():.2f}, {corr.max():.2f}]   <- note: NOT ~1.0")
print(f"  most coupled: {corr.idxmax()} ({corr.max():.2f});  "
      f"most buffered: {corr.idxmin()} ({corr.min():.2f})")

RNA-protein correlation: median=0.48, range=[-0.23, 0.92]   <- note: NOT ~1.0
  most coupled: VWA5A (0.92);  most buffered: ARPC1A (-0.23)


**Read this result.** The median correlation is well below 1 — most genes' protein levels only partly follow their RNA. That is the empirical reason you can't treat RNA as a stand-in for protein, and why integrating both layers adds information.

### 1.5 Per-layer differential signal
Each layer independently nominates candidates. Here the signal is the tumor-vs-normal effect size at RNA and protein, plus per-gene mutation frequency for the genomic layer.

In [7]:
if meta is not None:
    tcols = meta.index[meta == "tumor"]
    ncols = meta.index[meta == "normal"]
    rna_eff = (rna[tcols].mean(axis=1) - rna[ncols].mean(axis=1)).abs()
    prot_eff = (prot[tcols].mean(axis=1) - prot[ncols].mean(axis=1)).abs()
else:
    rna_eff = rna[samples].mean(axis=1).abs()
    prot_eff = prot[samples].mean(axis=1).abs()

scored = pd.DataFrame({"transcriptomic": rna_eff, "proteomic": prot_eff,
                       "genomic": mut_freq, "rna_prot_corr": corr})
scored.round(3).head()

,transcriptomic,proteomic,genomic,rna_prot_corr
A2M,0.060,0.494,0.033,0.349
A2ML1,0.954,3.147,0.016,NaN
AADACL2,0.499,0.396,0.008,NaN
AAED1,0.070,0.434,0.016,NaN
AAGAB,0.026,0.211,0.008,0.666


### 1.6 Multi-evidence score → ranked targets
Normalize each layer to a common scale and take the **equal-weighted** sum. This turns three noisy layers into one ranked list — the artifact everything downstream consumes.

In [8]:
scored["score"] = multi_evidence_score(
    scored, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
ranked_cptac = scored.sort_values("score", ascending=False)
print("Top CPTAC targets (worked example):")
print(ranked_cptac.head(6).round(3).to_string())
# Continuity check: TP53 / p53 (Module 1's P04637 / 1TUP) is a top breast-cancer hit.
print("\nTP53 rank:", list(ranked_cptac.index).index("TP53") + 1)

Top CPTAC targets (worked example):
         transcriptomic  proteomic  genomic  rna_prot_corr  score
SI                0.977      5.200    0.049            NaN  0.988
VWDE              1.263      1.424    0.049            NaN  0.979
MUC5B             0.566      3.441    0.082            NaN  0.978
SPHKAP            0.573      3.321    0.041            NaN  0.970
CEACAM5           0.813      3.315    0.033            NaN  0.970
RIMS2             1.240      1.676    0.033            NaN  0.968

TP53 rank: 108


### Part 1 recap
You integrated **at the sample level** because the data was **matched** — you could correlate RNA and protein within the same tumors. That was synthetic data, so the method is easy to see. Next we run it on real data, then you transfer it to unmatched data yourself.

---
# Part 2 — Find and load real CPTAC data *(together, in class)*

Now swap the synthetic files for the real thing. CPTAC breast data is open access (no data-use agreement) — we'll get it together in class. **Not graded.**

1. Go to **LinkedOmics** (`linkedomics.org`) → **CPTAC Breast Cancer (BRCA)**.
2. Download the gene-level **RNAseq** and **Proteome** matrices (gene × sample).
3. Save them into `data/` with genes as the row index and these exact names — transpose with `.T` first if samples are in rows:
   - `data/cptac_brca_rna.tsv`
   - `data/cptac_brca_protein.tsv`
   - `data/cptac_brca_mutation.tsv`  (one row per gene: mutation frequency or CNV magnitude)
4. `Kernel → Restart & Run All` and watch **Part 1** again — it now loads your real files instead of the demo. Compare: is the RNA–protein correlation still modest? Does TP53 still surface near the top?

*Wrinkle:* the real download has no tumor/normal labels, so ranking falls back to overall abundance rather than a tumor-vs-normal effect. That's fine for seeing the pipeline run on real data.

---
# Part 3 — Your assignment: Alzheimer's disease *(complete the `# TODO` cells)*

The three matrices you downloaded from Canvas are **gene-level summaries** from different cohorts — there are no shared samples to correlate, so you integrate **at the gene level.** You'll transfer the Part 1 workflow: harmonize → concordance → score → rank → export.

Your ranked target list (the CSV you export in 3.5) is what **Week 3** builds on, so it has to be clean. *Continuity:* expect **APOE** (`rs7412`) among your top hits.

> ### Note on Part 3: I used colon adenocarcinoma instead of Alzheimer's
>
> My professor said I could pick a different disease, so I did Part 3 on **colon
> adenocarcinoma (COAD)**. The method is the same as the assignment asks for. My
> three layers come from different cohorts with no patients in common, so I join
> them at the gene level, which is the whole point of Part 3.
>
> I did not change anything above this cell. `load_ad()`, `rank_percentile`,
> `multi_evidence_score` and `EQUAL_WEIGHTS` are all still the professor's code.
>
> **The three layers:**
>
> | Layer | Cohort | Comparison | Where it came from |
> |---|---|---|---|
> | Transcriptomic | TCGA-COAD (*Nature* 2012;487:330) | 471 tumours vs 41 adjacent normals | GDC STAR log2(TPM+1), via UCSC Xena |
> | Proteomic | CPTAC-2 Prospective Colon (Vasaikar, *Cell* 2019;177:1035, PMID 31031003) | 96 matched tumour/normal pairs | PNNL TMT gene-level log ratios, via LinkedOmics |
> | Genomic | The exome data from that same CPTAC study | 106 patients, tumour vs their own blood | WUSM GATK gene-level calls, via LinkedOmics |
>
> None of these come with a ready-made differential table, so I calculated each
> one myself in **`prepare_coad_data.py`** (Mann-Whitney for RNA, paired Wilcoxon
> for protein, and a length-corrected binomial test for the mutations). All the
> URLs, file names, sample counts and formulas are written down in
> **`data/PROVENANCE.md`**.
>
> One thing I checked before trusting any of it: my calculated protein log2FC
> matches the log2FC file the study published itself, at **Pearson r = 1.0000**
> over 6,420 shared genes.
>
> **The catch I have to be upfront about.** The RNA is from TCGA and the protein
> and mutations are from CPTAC, and no patient appears in both. CPTAC does have
> RNA-seq on its own 106 patients, but only on the tumours, with no adjacent
> normal samples, so there was no way to get a tumour vs normal RNA comparison
> out of it. That is why the RNA layer is TCGA, and it is why nothing below can
> say anything about individual patients.

In [9]:
# My loader for the colon files. I didn't touch the professor's load_ad()
# above - this is a separate function using the same three-table setup.
COAD_PATHS = {
    "tx": "data/coad_transcriptomics.tsv",   # gene, log2fc, pval, qval, ...
    "pr": "data/coad_proteomics.tsv",        # gene, log2fc, pval, qval, n_pairs
    "gw": "data/coad_genomics.tsv",          # gene, neglog10p, pval, qval, mut_freq, ...
}


def load_coad():
    """Read the three colon adenocarcinoma summary tables."""
    missing = [v for v in COAD_PATHS.values() if not os.path.exists(v)]
    if missing:
        raise FileNotFoundError(
            "Colon tables not found: " + ", ".join(missing) +
            "\nRun this once from the repo root to build them:\n"
            "    python prepare_coad_data.py\n"
            "It downloads the real TCGA and CPTAC files. See data/PROVENANCE.md.")
    return (pd.read_csv(COAD_PATHS["tx"], sep="\t"),
            pd.read_csv(COAD_PATHS["pr"], sep="\t"),
            pd.read_csv(COAD_PATHS["gw"], sep="\t"))


# Genes any colon cancer paper would mention. I use this to check the pipeline
# is behaving, the same way Part 1 checks for TP53. It is not a filter - I don't
# select or reweight anything based on this list.
KNOWN_COAD_GENES = ["APC", "TP53", "KRAS", "BRAF", "PIK3CA", "SMAD4", "CTNNB1",
                    "FBXW7", "TCF7L2", "SOX9", "ACVR2A", "RPL22", "NRAS",
                    "ERBB2", "MYC", "EGFR", "MSH2", "MLH1"]

# The gene I picked to investigate at the start of the project. I'm writing it
# here so it's clear I chose it before seeing any results, and that I never used
# it to steer the scoring.
TARGET_OF_INTEREST = "KRAS"   # UniProt P01116
print("Loader ready. Gene I set out to investigate:", TARGET_OF_INTEREST)

Loader ready. Gene I set out to investigate: KRAS


### 3.1 — TODO: load and inspect the three matrices
Call `load_ad()` and look at each table's columns and shape before you touch them.

In [10]:
# TODO 3.1 - load the three colon tables and look at them.
# The professor's line here was `tx, pr, gw = load_ad()`. I swapped it for my
# colon loader since I changed disease. Same three tables, same columns.
tx, pr, gw = load_coad()

for name, table in [("transcriptomics (TCGA-COAD)", tx),
                    ("proteomics (CPTAC-2 colon)", pr),
                    ("genomics (CPTAC-2 colon exomes)", gw)]:
    print(f"{name:34s} shape={table.shape}")
    print(f"{'':34s} columns={list(table.columns)}")

# I need to know the join key exists everywhere before I try merging.
print("\n'gene' column present in all three:",
      all("gene" in t.columns for t in (tx, pr, gw)))

print("\nFirst few rows of each:")
for name, table in [("tx", tx), ("pr", pr), ("gw", gw)]:
    print(f"\n-- {name} --")
    print(table.head(3).to_string(index=False))

transcriptomics (TCGA-COAD)        shape=(17611, 8)
                                   columns=['gene', 'log2fc', 'pval', 'qval', 'mean_tumor_log2tpm', 'mean_normal_log2tpm', 'n_tumor', 'n_normal']
proteomics (CPTAC-2 colon)         shape=(6709, 5)
                                   columns=['gene', 'log2fc', 'pval', 'qval', 'n_pairs']
genomics (CPTAC-2 colon exomes)    shape=(14706, 8)
                                   columns=['gene', 'neglog10p', 'pval', 'qval', 'mut_freq', 'n_mutated', 'cds_len', 'n_patients']

'gene' column present in all three: True

First few rows of each:

-- tx --
   gene    log2fc         pval         qval  mean_tumor_log2tpm  mean_normal_log2tpm  n_tumor  n_normal
   A1CF -1.140487 1.858317e-13 7.025938e-13            2.777139             3.917627      471        41
    A2M -1.361374 5.432866e-15 2.498126e-14            6.649796             8.011170      471        41
A2M-AS1 -0.459065 1.717535e-09 4.195386e-09            0.564779             1.023844      

### 3.2 — TODO: harmonize on the gene symbol → one joined table
Rename the effect/`pval` columns so RNA and protein don't collide, then merge all three on `gene`. Report how many genes survive the join (your first reality check).

In [11]:
# TODO 3.2 - build a single joined table `df`.
# Hint: rename before merging so columns don't clash.
tx = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr = pr.rename(columns={"log2fc": "prot_lfc", "pval": "prot_p"})

# My tables also have FDR q-values, and `qval` appears in both tx and pr, so if
# I don't rename those too pandas will quietly turn them into qval_x and qval_y.
# Same reason for renaming the genomic p-value.
tx = tx.rename(columns={"qval": "rna_q"})
pr = pr.rename(columns={"qval": "prot_q"})
gw = gw.rename(columns={"pval": "gen_p", "qval": "gen_q"})

# All three files use gene symbols, but symbols pick up stray spaces and case
# differences, and the TCGA ones were converted from Ensembl IDs. Cleaning them
# up first is what stops the merge from collapsing (the "merge gives 0 rows"
# problem in the troubleshooting table).
for t in (tx, pr, gw):
    t["gene"] = t["gene"].str.strip().str.upper()

# Inner join, so a gene has to be measured on all three layers to get scored.
df = tx.merge(pr, on="gene", how="inner").merge(gw, on="gene", how="inner")

print(f"Genes surviving the 3-way join: {len(df)}")
print(f"  transcriptomics on its own: {len(tx)}")
print(f"  proteomics on its own:      {len(pr)}")
print(f"  genomics on its own:        {len(gw)}")
print(f"  duplicate gene symbols after the join: {int(df['gene'].duplicated().sum())}")

# I wanted to know which layer was costing me the most genes.
print(f"\nRNA and genomics only (no protein needed): "
      f"{len(set(tx['gene']) & set(gw['gene']))}")
print("So proteomics is the bottleneck. Mass spec only measures a few thousand "
      "proteins in a tissue, so asking for protein evidence is what loses most "
      "of the genes.")

absent = [g for g in KNOWN_COAD_GENES if g not in set(df["gene"])]
print(f"\nKnown colon genes that didn't make the join: {absent}")
print("These are missing because the proteomics never measured them, not "
      "because the merge failed.")

Genes surviving the 3-way join: 4851
  transcriptomics on its own: 17611
  proteomics on its own:      6709
  genomics on its own:        14706
  duplicate gene symbols after the join: 0

RNA and genomics only (no protein needed): 10694
So proteomics is the bottleneck. Mass spec only measures a few thousand proteins in a tissue, so asking for protein evidence is what loses most of the genes.

Known colon genes that didn't make the join: ['APC', 'BRAF', 'FBXW7', 'TCF7L2', 'ACVR2A', 'MYC', 'MLH1']
These are missing because the proteomics never measured them, not because the merge failed.


### 3.3 — TODO: sign-agreement concordance
You can't correlate across samples here (nothing is matched), so concordance becomes **direction agreement**: does the gene move the *same way* at RNA and protein? Add a boolean `concordant` column. Watch for genes that are strong at RNA but flat/opposite at protein — those are the interesting discordant ones.

In [12]:
# TODO 3.3 - add a boolean `concordant` column: do RNA and protein point the same direction?
# Hint: compare np.sign(df["rna_lfc"]) with np.sign(df["prot_lfc"]).
df["concordant"] = np.sign(df["rna_lfc"]) == np.sign(df["prot_lfc"])
print(f"Sign-concordant genes: {df['concordant'].sum()} / {len(df)} "
      f"({100 * df['concordant'].mean():.1f}%)")
print(f"Discordant genes: {(~df['concordant']).sum()} "
      f"({100 * (~df['concordant']).mean():.1f}%)")

# Why direction and not correlation: TCGA and CPTAC are different patients, so
# there is nothing to correlate along. Agreeing on direction is the most I can
# get out of unmatched data.

# A gene can look discordant just because one layer is flat and noisy, so I
# also checked how it looks when both layers are individually significant.
both_sig = (df["rna_q"] < 0.05) & (df["prot_q"] < 0.05)
print(f"\nGenes significant on both layers (FDR < 0.05): {int(both_sig.sum())}")
print(f"  concordant: {int((both_sig & df['concordant']).sum())} "
      f"({100 * df.loc[both_sig, 'concordant'].mean():.1f}%)")
print(f"  discordant: {int((both_sig & ~df['concordant']).sum())}  "
      "<- these genuinely disagree, it isn't just noise")

# Is the signal mostly things the tumour gained or things it lost?
print("\nWhen both layers agree, which way do they go:")
print(f"  down at RNA and protein: "
      f"{int(((df.rna_lfc < 0) & (df.prot_lfc < 0)).sum())}")
print(f"  up at RNA and protein:   "
      f"{int(((df.rna_lfc > 0) & (df.prot_lfc > 0)).sum())}")

Sign-concordant genes: 3395 / 4851 (70.0%)
Discordant genes: 1456 (30.0%)

Genes significant on both layers (FDR < 0.05): 3115
  concordant: 2451 (78.7%)
  discordant: 664  <- these genuinely disagree, it isn't just noise

When both layers agree, which way do they go:
  down at RNA and protein: 1418
  up at RNA and protein:   1977


### 3.4 — TODO: multi-evidence score
Build a per-layer magnitude for each of the three layers, then score with `multi_evidence_score` and `EQUAL_WEIGHTS`. If you change the weights, justify it in your report.

In [13]:
# TODO 3.4 - score every gene.
# One magnitude per layer. They're on completely different scales, but
# rank_percentile inside multi_evidence_score puts them all on 0-1 first.
df["transcriptomic"] = df["rna_lfc"].abs()    # |log2FC| for RNA
df["proteomic"] = df["prot_lfc"].abs()        # |log2FC| for protein
df["genomic"] = df["neglog10p"]               # length-corrected mutation recurrence

df["score"] = multi_evidence_score(
    df, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)

print("Weights:", EQUAL_WEIGHTS, "(the course default - I kept these, and I "
      "explain why in the report)")
print(f"\nscore: min={df['score'].min():.3f} median={df['score'].median():.3f} "
      f"max={df['score'].max():.3f}")
print(f"NaN scores: {int(df['score'].isna().sum())}  (a NaN would sink a gene to "
      "the bottom of the sort without me noticing)")

print("\nThe three magnitudes before normalising. The scales are nothing like "
      "each other, which is why rank_percentile is needed:")
print(df[["transcriptomic", "proteomic", "genomic"]].describe().round(3).to_string())

Weights: {'transcriptomic': 0.3333333333333333, 'proteomic': 0.3333333333333333, 'genomic': 0.3333333333333333} (the course default - I kept these, and I explain why in the report)

score: min=0.021 median=0.495 max=0.992
NaN scores: 0  (a NaN would sink a gene to the bottom of the sort without me noticing)

The three magnitudes before normalising. The scales are nothing like each other, which is why rank_percentile is needed:
       transcriptomic  proteomic   genomic
count        4851.000   4851.000  4851.000
mean            0.631      0.369     0.368
std             0.640      0.389     1.295
min             0.000      0.000     0.000
25%             0.222      0.122     0.075
50%             0.473      0.257     0.185
75%             0.824      0.465     0.399
max             7.305      2.811    66.379


### 3.5 — TODO: rank, inspect, and export
Sort by score, take the top ~15, and **export the ranked table to `targets_ad.csv`** — this file is the hand-off to Week 3. Check whether known AD genes (APOE, TREM2, BIN1, CLU, PICALM …) are recovered, and look at any `concordant == False` genes in your top hits.

In [14]:
# TODO 3.5 - rank, look at the top 15, and export the CSV.
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))
top = ranked.head(15)

SHOW = ["rank", "gene", "rna_lfc", "prot_lfc", "neglog10p", "mut_freq",
        "concordant", "score"]

# The guide calls this targets_ad.csv. I named it for my disease instead.
OUT_CSV = "targets_coad.csv"
ranked.to_csv(OUT_CSV, index=False)

print(f"Top 15 colon targets (equal weights, {len(ranked)} genes ranked):")
print(top[SHOW].round(3).to_string(index=False))

print(f"\nExported the full ranked table to {OUT_CSV} "
      f"({len(ranked)} rows, {ranked.shape[1]} columns)")

# Did the pipeline find the genes a colon cancer paper would talk about?
print("\nKnown colon cancer genes, wherever they ended up:")
found = ranked[ranked["gene"].isin(KNOWN_COAD_GENES)]
print(found[SHOW].round(3).to_string(index=False))

# My gene of interest, reported wherever it landed.
row = ranked[ranked["gene"] == TARGET_OF_INTEREST]
if len(row):
    r = int(row["rank"].iloc[0])
    print(f"\n{TARGET_OF_INTEREST}: rank {r} out of {len(ranked)} "
          f"({100 * (1 - r / len(ranked)):.1f}th percentile)")
    print(row[SHOW + ["rna_q", "prot_q", "gen_q", "n_mutated"]].round(4).to_string(index=False))
    print(f"So {TARGET_OF_INTEREST} is one of the strongest genomic hits "
          f"(neglog10p={row['neglog10p'].iloc[0]:.1f}, mutated in "
          f"{100 * row['mut_freq'].iloc[0]:.0f}% of patients) but barely moves at "
          "RNA or protein. I didn't adjust anything to push it up. The report "
          "explains why a score built on abundance will always rank a gene like "
          "this low.")

# Discordant genes near the top, since 3.6 asks me to write about one.
disc = ranked.head(60).query("~concordant")
print(f"\nDiscordant genes in the top 60: {len(disc)}")
print(disc[SHOW + ["rna_q", "prot_q"]].round(3).to_string(index=False))

# What is the top of the list actually made of?
h = ranked.head(50)
print(f"\nTop 50 breakdown:  both layers down in tumour: "
      f"{int(((h.rna_lfc < 0) & (h.prot_lfc < 0)).sum())}   "
      f"both up: {int(((h.rna_lfc > 0) & (h.prot_lfc > 0)).sum())}   "
      f"discordant: {int((~h.concordant).sum())}")

Top 15 colon targets (equal weights, 4851 genes ranked):
 rank    gene  rna_lfc  prot_lfc  neglog10p  mut_freq  concordant  score
    1     CA2   -5.198    -2.157      1.966     0.047        True  0.992
    2     DES   -4.208    -2.791      1.490     0.057        True  0.989
    3     PTN   -2.297    -2.137      1.955     0.038        True  0.983
    4   HSPB6   -2.581    -2.274      1.308     0.028        True  0.979
    5    BCHE   -2.594    -1.753      1.485     0.066        True  0.978
    6    CFL2   -1.555    -2.202      1.973     0.038        True  0.972
    7   GREM2   -3.549    -1.352      1.257     0.028        True  0.971
    8    PRPH   -1.654    -2.258      1.490     0.057        True  0.970
    9  TSPAN7   -2.687    -1.001      2.040     0.047        True  0.967
   10 B3GALT5   -2.656    -1.385      1.117     0.038        True  0.965
   11     CR2   -1.968    -1.415      1.343     0.094        True  0.965
   12 COLEC12   -1.525    -1.874      1.477     0.075        True  

### A few extra checks

None of these change the ranking I exported above. I added them because three of
the four questions in 3.6 are much easier to answer with a number than with an
opinion:

1. Does the weighting actually matter? I re-scored with the genomic layer
   weighted more heavily and measured how much the list moved.
2. What did I lose by using two cohorts? CPTAC measured RNA and protein on the
   same 96 tumours, so inside that cohort I can do the Part 1 correlation and
   see what the unmatched version is missing.
3. Is this cohort actually "KRAS G12C"? I checked the variant calls rather than
   assuming.
4. Does the protein layer hold up in a completely different cohort?

In [15]:
# EXTRA 1 - does changing the weights change anything?
# My reasoning before running it: a somatic mutation is upstream of everything
# else and can't be caused by stromal contamination, whereas RNA and protein
# levels can be a downstream effect or just a different mix of cell types in the
# biopsy. That argues for giving the genomic layer more weight. So I tested it.
ALT_WEIGHTS = {"transcriptomic": 0.25, "proteomic": 0.25, "genomic": 0.50}

df["score_genomic_weighted"] = multi_evidence_score(
    df, ["transcriptomic", "proteomic", "genomic"], ALT_WEIGHTS)

alt = df.sort_values("score_genomic_weighted", ascending=False).reset_index(drop=True)
alt.insert(0, "alt_rank", np.arange(1, len(alt) + 1))

rho = df["score"].corr(df["score_genomic_weighted"], method="spearman")
print(f"Spearman correlation between the two rankings: {rho:.3f}")
print(f"Genes shared between the two top-15 lists: "
      f"{len(set(ranked.head(15)['gene']) & set(alt.head(15)['gene']))} / 15")

print("\nWhere doubling the genomic weight moves the genes I care about:")
moves = []
for g in ["KRAS", "TP53", "APC", "SOX9", "PIK3CA", "SMAD4", "RPL22"]:
    a = ranked.loc[ranked["gene"] == g, "rank"]
    b = alt.loc[alt["gene"] == g, "alt_rank"]
    if len(a) and len(b):
        moves.append((g, int(a.iloc[0]), int(b.iloc[0])))
print(pd.DataFrame(moves, columns=["gene", "rank_equal", "rank_genomic_weighted"])
      .to_string(index=False))

print("\nTop 10 with the genomic layer weighted at 0.5:")
print(alt.head(10)[["alt_rank", "gene", "rna_lfc", "prot_lfc", "neglog10p",
                    "score_genomic_weighted"]].round(3).to_string(index=False))
print("\nSo doubling the genomic weight hardly moves the top of the list at all. "
      "I think the reason is that rank_percentile squashes the genomic layer: "
      "neglog10p runs from 0 to 66, but converting to percentiles puts almost "
      "every gene into a narrow band, so there isn't much left for the weight to "
      "act on. I go into this in the report.")

# I put the alternative score in the CSV too, so whoever picks this up in Week 3
# can see the sensitivity check rather than taking my word for it.
ranked["score_genomic_weighted"] = ranked["gene"].map(
    df.set_index("gene")["score_genomic_weighted"])
ranked.to_csv(OUT_CSV, index=False)
print(f"\nRe-exported {OUT_CSV} with that column added.")

Spearman correlation between the two rankings: 0.941
Genes shared between the two top-15 lists: 14 / 15

Where doubling the genomic weight moves the genes I care about:
  gene  rank_equal  rank_genomic_weighted
  KRAS         821                    445
  TP53        1392                    716
  SOX9          40                     24
PIK3CA        2174                   1183
 SMAD4         895                    478
 RPL22        1209                    634

Top 10 with the genomic layer weighted at 0.5:
 alt_rank   gene  rna_lfc  prot_lfc  neglog10p  score_genomic_weighted
        1    CA2   -5.198    -2.157      1.966                   0.990
        2    DES   -4.208    -2.791      1.490                   0.984
        3    PTN   -2.297    -2.137      1.955                   0.983
        4   BCHE   -2.594    -1.753      1.485                   0.976
        5  HSPB6   -2.581    -2.274      1.308                   0.975
        6   CFL2   -1.555    -2.202      1.973                 


Re-exported targets_coad.csv with that column added.


In [16]:
# EXTRA 2 - what the two-cohort design costs me, measured inside CPTAC.
# CPTAC ran RNA-seq and proteomics on the same 96 tumours, so for those samples
# I can ask the Part 1 question: does protein follow RNA across patients? My
# Part 3 join can't ask this at all.
corr_path = "data/coad_cptac_matched_rna_protein_corr.tsv"
if os.path.exists(corr_path):
    mc = pd.read_csv(corr_path, sep="\t")
    print(f"CPTAC tumours measured on both layers: {int(mc['n_samples'].max())} "
          f"samples, {len(mc)} genes")
    print(f"Per-gene RNA-protein Spearman: median={mc['spearman_r'].median():.3f}, "
          f"range=[{mc['spearman_r'].min():.2f}, {mc['spearman_r'].max():.2f}]"
          "   <- nowhere near 1.0")
    print(f"  genes where protein follows RNA (r > 0.5): "
          f"{int((mc['spearman_r'] > 0.5).sum())} "
          f"({100 * (mc['spearman_r'] > 0.5).mean():.1f}%)")
    print(f"  genes where protein goes the other way (r < 0): "
          f"{int((mc['spearman_r'] < 0).sum())} "
          f"({100 * (mc['spearman_r'] < 0).mean():.1f}%)")
    print(f"  most coupled: {mc.loc[mc['spearman_r'].idxmax(), 'gene']} "
          f"({mc['spearman_r'].max():.2f});  most buffered: "
          f"{mc.loc[mc['spearman_r'].idxmin(), 'gene']} ({mc['spearman_r'].min():.2f})")
    print("\nThis is the same result as Part 1, just on real colon data: the "
          "median is well below 1, so RNA doesn't stand in for protein. It's also "
          "the resolution my Part 3 ranking doesn't have - my concordant column "
          "turns this whole range into a single True/False.")
else:
    print(f"{corr_path} is missing - run `python prepare_coad_data.py`.")


# EXTRA 3 - is this actually a KRAS G12C cohort? Checking instead of assuming.
kras_path = "data/coad_kras_variant_spectrum.tsv"
if os.path.exists(kras_path):
    kv = pd.read_csv(kras_path, sep="\t")
    print("\nKRAS mutations found in the 106 CPTAC colon patients:")
    print(kv.to_string(index=False))
    g12c = kv.loc[kv["variant"].str.contains("G12C"), "n_patients_mutated"]
    print(f"\nG12C shows up in {int(g12c.iloc[0]) if len(g12c) else 0} of 106 "
          "patients. This cohort is mostly G12D and G12V, so it would be wrong "
          "to call it a G12C dataset. The three layers I scored are gene-level "
          "anyway and don't know anything about which allele it is.")
else:
    print(f"\n{kras_path} is missing - run `python prepare_coad_data.py`.")

CPTAC tumours measured on both layers: 96 samples, 7069 genes
Per-gene RNA-protein Spearman: median=0.336, range=[-0.52, 0.91]   <- nowhere near 1.0
  genes where protein follows RNA (r > 0.5): 1721 (24.3%)
  genes where protein goes the other way (r < 0): 450 (6.4%)
  most coupled: SERPINB5 (0.91);  most buffered: RERG (-0.52)

This is the same result as Part 1, just on real colon data: the median is well below 1, so RNA doesn't stand in for protein. It's also the resolution my Part 3 ranking doesn't have - my concordant column turns this whole range into a single True/False.

KRAS mutations found in the 106 CPTAC colon patients:
     variant  n_patients_mutated  pct_of_cohort
 KRAS_p.G12D                  11          10.38
 KRAS_p.G12V                   7           6.60
 KRAS_p.G13D                   4           3.77
KRAS_p.A146T                   3           2.83
 KRAS_p.G12C                   2           1.89
KRAS_p.K117N                   2           1.89
 KRAS_p.Q61H             

In [17]:
# EXTRA 4 - which of these could you actually drug?
# 43 of my top 50 are genes the tumour LOST. That's fine as a biomarker signal
# but it's not much use if you want to inhibit something. So here I keep the
# ranking exactly as it is and just read off the genes that went UP on both
# layers. This is a filter on the output, not a different score.
up = ranked[(ranked["rna_lfc"] > 0) & (ranked["prot_lfc"] > 0)]
print(f"{len(up)} of {len(ranked)} ranked genes are up on both layers "
      f"({100 * len(up) / len(ranked):.1f}%)")
print("\nBest-scoring genes that are up at both RNA and protein:")
print(up.head(15)[["rank", "gene", "rna_lfc", "prot_lfc", "neglog10p",
                   "mut_freq", "score"]].round(3).to_string(index=False))

1977 of 4851 ranked genes are up on both layers (40.8%)

Best-scoring genes that are up at both RNA and protein:
 rank   gene  rna_lfc  prot_lfc  neglog10p  mut_freq  score
   22   PLAU    2.102     0.971      1.166     0.047  0.950
   27 TMEM97    1.730     0.768      3.514     0.057  0.945
   40   SOX9    2.061     0.583     10.929     0.179  0.933
   44  DHCR7    1.513     0.753      1.472     0.057  0.930
   49  REEP6    1.449     0.874      1.029     0.028  0.926
   53  DDX27    1.163     0.770      4.340     0.132  0.920
   55  SULF1    1.722     0.643      1.151     0.075  0.919
   56 LRRC15    1.226     0.903      1.116     0.057  0.917
   61   TPX2    2.036     0.727      0.762     0.057  0.915
   62  MMP11    3.602     0.486      1.918     0.066  0.914
   75  VSNL1    2.319     0.922      0.543     0.019  0.908
   80   P3H1    1.200     0.770      0.962     0.066  0.902
   85  DPEP1    4.952     0.851      0.462     0.028  0.899
   87   ANLN    2.134     0.891      0.489     

In [18]:
# EXTRA 5 - does the protein layer hold up in a different cohort?
# My r = 1.0000 check only proves I did the arithmetic the same way CPTAC did.
# It doesn't prove the biology replicates. So I compared against a completely
# separate study: supplementary table S6B of "Comprehensive Proteogenomic
# Profiling Reveals the Molecular Characteristics of Colorectal Cancer at
# Distinct Stages of Progression" (Cancer Research 2024;84:2888-2910), a Chinese
# cohort with 28 paired tumour/normal samples measured label-free instead of TMT.
fudan_path = "data/coad_independent_proteomics_fudan.tsv"
if os.path.exists(fudan_path):
    fu = pd.read_csv(fudan_path, sep="\t")
    cmp = df.merge(fu, on="gene")
    print(f"Proteins measured in both CPTAC and the Fudan cohort: {len(cmp)}")
    pear = np.corrcoef(cmp["prot_lfc"], cmp["log2fc_fudan"])[0, 1]
    spear = stats.spearmanr(cmp["prot_lfc"], cmp["log2fc_fudan"]).statistic
    agree = (np.sign(cmp["prot_lfc"]) == np.sign(cmp["log2fc_fudan"])).mean()
    print(f"  Pearson r  = {pear:.3f}")
    print(f"  Spearman r = {spear:.3f}")
    print(f"  same direction in both cohorts: {100 * agree:.1f}%")

    sig = cmp[cmp["prot_q"] < 0.05]
    sig_agree = (np.sign(sig["prot_lfc"]) == np.sign(sig["log2fc_fudan"])).mean()
    print(f"\n  restricted to genes significant in CPTAC (n={len(sig)}): "
          f"{100 * sig_agree:.1f}% agree on direction, "
          f"Pearson r = {np.corrcoef(sig['prot_lfc'], sig['log2fc_fudan'])[0, 1]:.3f}")

    print("\nAbout two thirds of proteins move the same way in both cohorts. "
          "That's a moderate agreement, not a great one, but it's roughly what I'd "
          "expect between TMT and label-free measurements on different patients in "
          "different countries. It does mean I shouldn't read too much into any "
          "single protein's fold change on its own.")
else:
    print(f"{fudan_path} is missing. It needs the journal spreadsheet saved into "
          "data/raw/ - see prepare_coad_data.py.")

Proteins measured in both CPTAC and the Fudan cohort: 2960
  Pearson r  = 0.451
  Spearman r = 0.481
  same direction in both cohorts: 65.5%

  restricted to genes significant in CPTAC (n=2435): 68.3% agree on direction, Pearson r = 0.483

About two thirds of proteins move the same way in both cohorts. That's a moderate agreement, not a great one, but it's roughly what I'd expect between TMT and label-free measurements on different patients in different countries. It does mean I shouldn't read too much into any single protein's fold change on its own.


### 3.6 — Interpretation (write-up, goes in your report)
Answer these in your 3–4 page report — this is the 40% interpretation payload:

1. **Weighting.** You used equal weights. Argue for keeping them equal *or* for up-weighting a layer (e.g. GWAS as germline/causal-leaning vs. transcriptomics as possibly downstream). There's no single right answer — only reasoned vs. unreasoned.
2. **Top targets.** Which known AD genes did you recover (APOE, TREM2, BIN1, CLU, PICALM …)? Any non-obvious hit worth a second look?
3. **Read a discordant gene.** Pick a `concordant == False` gene in your top hits (strong RNA/GWAS, flat protein). What biology could explain RNA and protein disagreeing?
4. **Limitation.** This was **gene-level, cross-cohort** integration — unmatched — so you *cannot* make per-patient claims. Contrast this with the matched CPTAC case from Part 1.

### The 3.6 write-up

I answered the four interpretation questions in **`Module2_Part3_Report.md`** in
the repo root. Every number in it comes from the cells above or from
`prepare_coad_data.py`, and `data/PROVENANCE.md` says where each input file came
from.

---
### Submit (Part 3 only)
1. `Kernel → Restart & Run All` — confirm the whole notebook runs top to bottom.
2. Commit **this notebook**, **`targets_ad.csv`**, and a short **README** (your name + anything that didn't work) to your `biot6900` repo.
3. Push, confirm the files appear on github.com, then **post your repo link on Canvas.**

Grading follows the course 60 / 40 split — 60 execution, 40 interpretation & communication. Partial credit for a correct approach even with minor technical errors: if a step wouldn't run, say what you were trying to do and what happened.